In [ ]:
import scanpy as sc
# from matplotlib import rcParams
# import pyMuTrans as pm
# import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

In [ ]:
color_palette = sns.color_palette() # setup color palette

In [ ]:
# Step 1: Select the input dataset

# adata_cutoff = sc.read_h5ad("adata_LacZ_ind_0919.h5ad")
# adata_cutoff = sc.read_h5ad("adata_Shroom3_ind_0919.h5ad")

In [ ]:
# Step 2: Select the cutoff rate "r"
r = 0.6

In [ ]:
# Step 3: Compute new meta data

En_cutoff = adata_cutoff.obs['entropy']
En_cutoff = pd.DataFrame(En_cutoff)

# the cutoff value = maximal*r (above cutoff value: transition cell)
cutoff = En_cutoff.max()*r
En_cutoff[En_cutoff.entropy > cutoff.entropy]=2
En_cutoff[(En_cutoff.entropy < cutoff.entropy) | (En_cutoff.entropy == cutoff.entropy)]=0

En_cutoff = pd.to_numeric(En_cutoff.entropy)
En_cutoff = En_cutoff.astype('category')
En_cutoff.cat.categories = ['Stable', 'Transition']
# the new meta data is named "En_cutoff"
adata_cutoff.obs['En_cutoff'] = En_cutoff

# visualize the result on umap (if not satisfy, go back and change the cutoff rate)
sc.pl.umap(adata_cutoff, color=['En_cutoff'], vmax = 'p95', palette = color_palette)

In [ ]:
# Step 4: Save the processed dataset 

# adata_coL = adata_cutoff
# adata_coS = adata_cutoff

In [ ]:
# After saving both two datasets, we can move to step 5
# Step 5: Compute the proportions of transition/stable cells in each Seurat cluster

label = np.asarray(adata_coL.obs['seurat_clusters']).ravel()
label = np.unique(label)
n = len(label)
L =  np.zeros((n,2)) # the results of LacZ
S =  np.zeros((n,2)) # the results of Shroom3

for i in range(0,n-1):
    s = label[i]
    adata_SL = adata_coL[adata_coL.obs['seurat_clusters'] == s]
    adata_SS = adata_coS[adata_coS.obs['seurat_clusters'] == s]
    total_L = len(adata_SL.obs['entropy'])
    total_S = len(adata_SS.obs['entropy'])
    tr_L = adata_SL.obs['En_cutoff'][adata_SL.obs['En_cutoff'] == 'Transition']
    st_L = adata_SL.obs['En_cutoff'][adata_SL.obs['En_cutoff'] == 'Stable']
    tr_S = adata_SS.obs['En_cutoff'][adata_SS.obs['En_cutoff'] == 'Transition']
    st_S = adata_SS.obs['En_cutoff'][adata_SS.obs['En_cutoff'] == 'Stable']
    L[i,0] = len(tr_L)/total_L # first column: proportions of transition cell
    L[i,1] = len(st_L)/total_L # second column: proportions of stable cell
    S[i,0] = len(tr_S)/total_S
    S[i,1] = len(st_S)/total_S

In [ ]:
# Basic visualization 

a = np.array(L[:,0]*100)
b = np.array(S[:,0]*100)

plt.barh(range(len(a)), a) # right side: LacZ
plt.barh(range(len(b)), -b) # left side: Shroom3